# Phase 2 — compact predictive diagnostics and RQ4 handoff

**Research question (RQ3).** A generative LLM, used as a *pairwise ranker*, looks at two
documents and says which better answers a query. The pooled DL19 + DL20 top-10 population
contains 4,330 sampled pairs; the stable, decisive analysis subsets contain 3,158 Qwen
and 2,931 Flan pairs. Classical *retrieval axioms* also vote on each pair (e.g. "prefer
the document with more query‑term occurrences"). RQ3 asks:

> **What incremental predictive value does the existing axiom battery have on held-out
> queries, and which observations are confirmed measurements versus developmental leads?**

This compact companion (1) measures existing-battery prediction, (2) tests incremental
covariate associations, and (3) maps rank-gap scope. Out-of-fold errors are diagnostic
groundwork for RQ4, not evidence of a residual mechanism or the sole source of candidates.

Everything reads cached `results/rq3_decomposition/` artefacts—there are no model calls.
Regeneration is documented in `docs/phase2-implementation.md`; the scientific protocol
and decision record are `docs/phase2-design.md`. Two models are analysed:
**Qwen 3.6 35B** (primary) and **Flan‑T5‑large** (model contrast).
Both see the same pooled DL19+DL20 top-10 pairs, so the contrast is not an independent
replication sample.

**Units and guardrails.** Uncertainty and CV splits are grouped by query; decisive pairs
are the prediction units. Position-inconsistent pairs are excluded by the Phase 0
measurement definition. The decomposition is predictive, not causal or cognitive:
accuracy, log-loss reduction, and cluster structure do not reveal an LLM mechanism.
All cluster and covariate interpretations are hypothesis-generating unless separately
validated.

In [ ]:
import json

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd

from axiomrank.paths import results_dir

# Shared palette (matches the Phase 1 notebook).
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, BASE, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
PRIMARY, CONTRAST = "models/qwen3.6-35B-A3B-AWQ", "google/flan-t5-large"
MODELS = [PRIMARY, CONTRAST]
MODEL_DIR = {PRIMARY: "models__qwen3.6-35B-A3B-AWQ", CONTRAST: "google__flan-t5-large"}
MODEL_COLOR = {PRIMARY: "#2a78d6", CONTRAST: "#1baf7a"}
MODEL_LABEL = {PRIMARY: "Qwen 3.6 35B", CONTRAST: "Flan-T5-large"}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": INK2,
    "axes.edgecolor": BASE, "font.size": 11, "axes.titlesize": 12,
})

CELL = results_dir("rq3_decomposition") / "pooled_top10" / "metrics"


def load(model, filename):
    path = CELL / MODEL_DIR[model] / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing Phase 2 artefact: {path}")
    return pd.read_csv(path) if filename.endswith(".csv") else json.loads(path.read_text())


## 1. Existing-battery prediction

**Method.** We predict the LLM's decisive verdict (which document it preferred) from *all*
axiom votes at once, with an L2‑regularised logistic regression under **query‑grouped
5‑fold cross‑validation** — so every number below is measured on queries the model never saw
while fitting. We compare it with the out-of-fold null:

* **Fold-specific null** — accuracy of predicting the training-fold majority label.
Order-swap consistency is reported separately. It is not an accuracy ceiling or estimate
of aleatoric noise because swapping order changes the prompt systematically.

**Two predictive summaries:**

* **CV accuracy** — fraction of verdicts predicted correctly out‑of‑fold. Intuitive, but a
  pair called right at p = 0.51 counts the same as one called at p = 0.99.
* **Normalised log-loss gain** — the relative reduction in out-of-fold cross-entropy
  against the class-prior null, $1-CE_{model}/CE_{null}$. It rewards calibrated
  probabilities and does not hinge on the 0.5 threshold. It is **not** a literal fraction
  of behaviour, variance, cognition, or mechanism explained; values can be negative.

**How to read the figure.** Each row is one model; the line runs from the training-fold
class-prior null (grey) to combined-model out-of-fold accuracy (colour).

In [ ]:
rows = []
for m in MODELS:
    d = load(m, "decomposition.json"); lex, sem = d["lexical"], d["lexical_semantic"]
    rows.append({
        "model": m, "n": lex["n_decisive_pairs"], "base": lex["cv_null_accuracy"],
        "cv_acc": lex["cv_accuracy"], "gain": lex["accuracy_gain_over_base"],
        "auc": lex["cv_auc"],
        "norm_logloss_gain": lex["information"]["pseudo_r2"],
        "nl_headroom": lex["nonlinear_headroom"], "sem_dcv": sem["cv_accuracy"] - lex["cv_accuracy"],
    })
dec = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8.6, 2.6))
for y, r in enumerate(dec.itertuples()):
    ax.plot([r.base, r.cv_acc], [y, y], color=GRID, linewidth=2.5, zorder=1)
    ax.scatter(r.base, y, s=70, color=MUTED, zorder=2)
    ax.scatter(r.cv_acc, y, s=95, color=MODEL_COLOR[r.model], zorder=3)
ax.set_yticks(range(len(dec)), [MODEL_LABEL[r.model] for r in dec.itertuples()])
ax.invert_yaxis(); ax.set_xlim(0.50, 0.90)
ax.set_xlabel("accuracy")
ax.set_title("Predictive gain: out-of-fold null → axiom model", loc="left")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True); ax.tick_params(length=0)
legend = [
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor=MUTED, markeredgecolor=MUTED,
           markersize=9, label="base rate (guess majority class)"),
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor=INK2, markeredgecolor=INK2,
           markersize=9, label="combined axiom model (CV accuracy)"),
]
ax.legend(handles=legend, loc="lower right", frameon=False, fontsize=8.5)
plt.tight_layout(); plt.show()

dec.assign(model=lambda x: x.model.map(MODEL_LABEL)).set_index("model").round(4)


**Reading it.** The combined model exceeds the majority baseline by ~0.07–0.08 accuracy
and normalised OOF log-loss gain is **0.060 (Qwen) / 0.076 (Flan)**. These are modest
predictive improvements relative to the fold-specific null, not fractions of behaviour or
mechanism.
The gradient‑boosted variant (`nl_headroom`) adds little in this implementation, providing
no detected large interaction gain under its chosen capacity; it does not rule out other
nonlinear models or representations. Adding
WordNet semantics (`sem_dcv` < 0) *lowers* accuracy, so the Phase 1 RQ2 null carries into the
pool. Order sensitivity is a separate outcome; no reliability, noise, or reducibility
partition is identified. The incremental analysis below tests covariate association. Per-collection
fits provide a robustness check:

In [ ]:
rows = []
for m in MODELS:
    for c in ["dl19_top10", "dl20_top10"]:
        cd = load(m, f"by_collection/{c}/decomposition.json")["lexical"]
        rows.append({"model": MODEL_LABEL[m], "cell": c, "base": round(cd["base_rate"], 3),
                     "cv_acc": round(cd["cv_accuracy"], 3),
                     "gain": round(cd["accuracy_gain_over_base"], 3)})
pd.DataFrame(rows).set_index(["model", "cell"])


## 2. Incremental covariate diagnostic

Patterns among out-of-fold errors can help prioritise RQ4 candidates if they generalise to
held-out queries. The following test detects incremental predictive structure; it cannot
separate every source of error, establish that a pattern is axiomatisable, or determine
whether RQ4 should be pursued.

**Incremental model.** For each pair we use raw covariates—signed *doc₁ − doc₂*
differences not included as identical columns in the axiom model:

| covariate | meaning |
|---|---|
| `d_len`  | difference in document length (verbosity) |
| `d_qcov` | difference in query‑term coverage |
| `d_rank`, `d_score` | difference in BM25 rank / score (first‑stage strength) |

Length and query coverage are related to LNC/AND-style concepts and may correlate with
axiom votes; they are not guaranteed to be orthogonal or conceptually novel. We ask
whether these covariates can predict the LLM's verdict **on top of what the axioms
already say** (we condition on the axiom model's own prediction). That improvement is the
**lift**. If the lift's 95 % query‑bootstrap CI is entirely above zero, the covariates carry
held-out predictive signal conditional on this fitted axiom model. It does not by itself
prove a new axiom, causal mechanism, or complete separation from existing axioms.

We report two covariate sets: **`content`** (`d_len`, `d_qcov`—text properties that may
help prioritise content-axiom candidates) and **`all`** (adds rank/score strength). The
content-only comparison is the relevant diagnostic for candidate leads; first-stage
strength is predictive metadata rather than an admissible content-axiom seed.

**How to read the figure.** Each row is one *model × covariate set*. The bar is the 95 % CI,
the marker is the lift. A **filled** marker with its whole bar right of the dotted zero line
= interval excludes zero; an **open** marker straddling zero = no detected lift under this
analysis. The interval does not classify observations as 'signal' or 'noise'.

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 3.2))
labels, y = [], 0
for m in MODELS:
    r = load(m, "residual_model.json")
    for setname, key in [("content", "content_only"), ("all", "all_covariates")]:
        v = r[key]; above = v["lift_ci_lo"] > 0
        ax.plot([v["lift_ci_lo"], v["lift_ci_hi"]], [y, y], color=GRID, linewidth=3, zorder=1)
        ax.scatter(v["lift"], y, s=90, zorder=3, color=MODEL_COLOR[m],
                   facecolor=MODEL_COLOR[m] if above else SURFACE,
                   edgecolor=MODEL_COLOR[m], linewidth=1.8)
        ax.text(0.102, y, f'{v["lift"]:+.3f}  [{v["lift_ci_lo"]:+.3f}, {v["lift_ci_hi"]:+.3f}]',
                va="center", ha="left", fontsize=8.5, color=INK2)
        labels.append(f"{MODEL_LABEL[m]} · {setname}"); y += 1
ax.axvline(0, color=MUTED, linewidth=1.2, linestyle=(0, (2, 2)), zorder=2)
ax.set_yticks(range(len(labels)), labels); ax.invert_yaxis(); ax.set_xlim(-0.03, 0.185)
ax.set_xlabel("lift in accuracy over the axiom-only baseline")
ax.set_title("Incremental covariate lift (95% query-bootstrap CI)", loc="left")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, color=GRID, linewidth=0.8); ax.set_axisbelow(True); ax.tick_params(length=0)
legend = [
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor=INK2, markeredgecolor=INK2,
           markersize=9, label="filled: interval excludes zero"),
    Line2D([0], [0], marker="o", linestyle="none", markerfacecolor=SURFACE, markeredgecolor=INK2,
           markersize=9, label="open: CI includes 0 (unresolved)"),
    Line2D([0], [0], color=MUTED, linestyle=(0, (2, 2)), label="dashed: zero lift"),
]
ax.legend(handles=legend, loc="upper center", bbox_to_anchor=(0.5, -0.20), ncol=3,
          frameon=False, fontsize=8.5)
fig.subplots_adjust(left=0.22, right=0.97, bottom=0.28, top=0.85); plt.show()


**Reading it.** The corrected same-fold comparison does not confirm incremental content
signal: Qwen is **+0.011, query-bootstrap CI [−0.012, +0.034]**, and Flan is
**+0.006 [−0.009, +0.021]**. Qwen's broader `all` set is positive (**+0.029
[+0.003, +0.056]**), but it adds BM25 rank and score—useful predictive metadata, not
admissible content axioms. The content-only analysis therefore supplies **no confirmed
incremental mechanism**. The clusters below remain developmental descriptions and do not
decide whether RQ4 proceeds.

**Exploratory error summary.** We cluster out-of-fold errors by
their covariates and inspect each cluster's centroid. Both models produce clusters with
large mean differences along **document length** and **query-term coverage**. This
cross-model resemblance suggests two error-analysis leads for RQ4. Candidate formulation must
also be grounded in retrieval theory and prior literature; cluster labels, centroids, and
counts require stability analysis and qualitative passage inspection before they can
support an axiom claim.

**How to read the figure.** Each bubble is a residual cluster, placed at its mean length
difference (x) and query‑coverage difference (y); bubble area ∝ number of pairs. A bubble far
from the origin marks a cluster with a large average signed difference; it does not by
itself establish why the pair was misclassified or whether the direction generalises.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), sharex=True, sharey=True)
for ax, m in zip(axes, MODELS):
    cl = load(m, "residual_clusters.csv")
    ax.axhline(0, color=GRID, lw=1, zorder=1); ax.axvline(0, color=GRID, lw=1, zorder=1)
    ax.scatter(cl["mean_d_len"], cl["mean_d_qcov"], s=cl["size"] * 1.2, zorder=3,
               color=MODEL_COLOR[m], alpha=0.7, edgecolor=INK2, linewidth=0.9)
    for r in cl.itertuples():
        ax.annotate(f"n={r.size}", (r.mean_d_len, r.mean_d_qcov), textcoords="offset points",
                    xytext=(0, 14), ha="center", fontsize=8, color=INK2)
    ax.set_title(MODEL_LABEL[m], loc="left")
    ax.set_xlabel("mean d_len   (doc₁ − doc₂ length →)")
    ax.spines[["top", "right"]].set_visible(False); ax.tick_params(length=0)
axes[0].set_ylabel("mean d_qcov   (query coverage →)")
fig.suptitle("Exploratory OOF-error clusters: length and coverage coordinates",
             x=0.5, y=1.02, fontsize=12.5)
plt.tight_layout(); plt.show()


## 3. Rank-gap scope diagnostic

A developmental hypothesis was that, if BM25 rank separation is a useful difficulty proxy
and classical axioms describe widely separated pairs, joint fidelity would rise with
absolute rank gap. BM25 gap is neither ground-truth difficulty nor an axiom-validity
measure, and no particular shape is required for pipeline validity.

**BM25 as the yardstick.** We also plot **BM25 used directly as a pairwise predictor** — it
"prefers" the better first‑stage-ranked document — scored against the LLM on the *same*
decisive pairs. This is the Phase 1 effectiveness reference and a **same-frame
comparator**. Because the bins are defined from BM25 ranks, a BM25 trend is
partly structurally favoured and cannot validate the axiom computation pipeline by itself.
It does show whether LLM–BM25 alignment varies over the same observed gap range.

**How to read the figure.**

* **Left — top‑10 pairs (gaps 1–9).** **Solid + round** = the combined axiom model's
  accuracy; **dash‑dot + square** (same colour) = **BM25‑as‑predictor** accuracy; **dashed**
  = the LLM's *decisive rate* (how often it gives a firm, order‑consistent verdict at all);
  grey **dotted** = chance (0.5).
* **Right — the wide‑gap `uniform50` control (full rank‑gap range).** Solid+round = the
  combined model's accuracy and dash‑dot+square = BM25, over gaps far wider than the top‑10
  ever reaches.

The comparison asks whether either predictor's fidelity changes across the same bins.
No bin-level uncertainty intervals are plotted, so local fluctuations are descriptive.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for m in MODELS:
    g = load(m, "gap_gradient.csv").drop_duplicates("gap_bin").sort_values("gap_bin")
    ax[0].plot(g["gap_bin"], g["joint_cv_accuracy"], marker="o", color=MODEL_COLOR[m], zorder=3)
    ax[0].plot(g["gap_bin"], g["bm25_accuracy"], marker="s", markersize=4,
               linestyle=(0, (3, 1, 1, 1)), color=MODEL_COLOR[m], alpha=0.8, zorder=2)
    ax[0].plot(g["gap_bin"], g["decisive_rate"], linestyle=(0, (4, 3)), color=MODEL_COLOR[m],
               alpha=0.55, zorder=2)
    u = load(m, "gap_gradient_uniform.csv").drop_duplicates("gap_bin").sort_values("gap_bin")
    ax[1].plot(u["gap_bin"], u["joint_cv_accuracy"], marker="o", color=MODEL_COLOR[m], zorder=3)
    ax[1].plot(u["gap_bin"], u["bm25_accuracy"], marker="s", markersize=4,
               linestyle=(0, (3, 1, 1, 1)), color=MODEL_COLOR[m], alpha=0.8, zorder=2)
for a in ax:
    a.axhline(0.5, color=MUTED, lw=1.1, linestyle=(0, (1, 2)), zorder=1)
    a.set_xlabel("BM25 rank gap  |rank₁ − rank₂|")
    a.spines[["top", "right"]].set_visible(False)
    a.yaxis.grid(True, color=GRID, lw=0.8); a.set_axisbelow(True); a.tick_params(length=0)
ax[0].set_title("top-10 pairs (gaps 1–9)", loc="left")
ax[1].set_title("uniform50 control (full rank-gap range)", loc="left")
ax[0].set_ylabel("accuracy / rate")

style_legend = [
    Line2D([0], [0], color=INK2, marker="o", label="combined-model accuracy"),
    Line2D([0], [0], color=INK2, marker="s", markersize=5, linestyle=(0, (3, 1, 1, 1)),
           label="BM25-as-predictor accuracy"),
    Line2D([0], [0], color=INK2, linestyle=(0, (4, 3)), label="LLM decisive rate"),
    Line2D([0], [0], color=MUTED, linestyle=(0, (1, 2)), label="chance (0.5)"),
]
color_legend = [Line2D([0], [0], color=MODEL_COLOR[m], marker="o", label=MODEL_LABEL[m])
                for m in MODELS]
ax[0].legend(handles=style_legend, loc="lower right", frameon=False, fontsize=8.5)
ax[1].legend(handles=color_legend, loc="lower right", frameon=False, fontsize=8.5)
plt.tight_layout(); plt.show()

**Reading it.** Within top‑10, combined-model accuracy and the LLM decisive rate both rise
with gap. Their co-movement is compatible with presentation robustness contributing to
the pattern, but the plot does not decompose or identify that contribution. BM25-as-
predictor fidelity also rises (≈0.55 → 0.76 for Qwen; 0.55 → 0.81 for Flan) and exceeds
the combined model in the widest top-10 bins.

The `uniform50` control makes it unmistakable. There BM25 rises steeply to **~0.84 (Qwen) /
~0.78 (Flan)** as the gap widens, while the combined axiom model remains roughly flat
around 0.55 over most bins. This rejects the specific expectation of a broad, monotonic
wide-gap gain for the fitted axiom model in these data.

**Conclusion:** the hypothesised broad wide-gap gain is **not observed for the combined
axiom model**. The BM25 comparator shows that LLM–BM25 alignment changes across
the bins, but—because the bins themselves derive from BM25—it cannot rule out axiom
implementation, sampling, or calibration explanations. This maps the fitted battery's
scope rather than passing or failing the pipeline. Rank/score features remain difficulty
covariates rather than candidate content axioms under the study design.

## Summary and bounded hand-off

Putting the three pieces together:

1. **§1 — modest predictive gain.** The combined axiom model gains ~0.07–0.08 accuracy
   and 0.060–0.076 normalised log-loss relative to the fold-specific null. These are
   predictive comparisons only.
2. **§2 — no confirmed content mechanism.** Neither model's pooled content-only lift
   excludes zero. Qwen has broader rank/score-conditioned signal, but that is not evidence
   for a new content axiom. Length and query coverage remain exploratory hypotheses.
3. **§3 — scoped rank-gap result.** The combined model does not show the hypothesised
   broad wide-gap gradient. BM25 changes across BM25-defined bins, which is useful context
   but not an independent pipeline validation.

Phase 2 supplies an evidence-graded handoff to RQ4. **RQ4 is the
thesis's main contribution**, with these diagnostics serving as developmental groundwork,
not confirmatory proof. Candidate axioms should be derived from retrieval theory and prior
literature, informed—but not dictated—by error-analysis leads such as length and query
coverage. Each candidate then requires an iterative cycle:
formalise on development data, freeze its definition and decision rule, evaluate on
held-out queries or collections, inspect failure cases, and retain it only if the effect
replicates without post-hoc threshold tuning. Full numerical results and caveats belong
in `docs/phase2-design.md` §6 and `docs/phase3-design.md`.